# Chương 3 — Demo Hai Ứng Dụng `negFIN` (Local — Pure Julia)

Phiên bản chạy trực tiếp trên **máy local** — **không cần setup/config**, gọi thẳng mã nguồn `applications.jl` và `negFIN` tự viết của nhóm.

### Hướng dẫn chạy trên máy Local
1. Đảm bảo đã có sẵn dữ liệu `data/chapter3/groceries.csv` và `digits8x8.csv` trong thư mục dự án.
2. Chọn **Kernel: Julia** trong VS Code hoặc Jupyter.
3. Bấm **Run All** để chạy kiểm thử Chương 3 và cả 2 ứng dụng (*Market Basket* & *Bag of Visual Words*).

## Phần 1: Nạp Module `Applications` Từ Thư Mục Dự Án

In [ ]:
using Pkg: Pkg

function find_project_root()::String
    dir = pwd()
    for _ in 1:4
        if isfile(joinpath(dir, "Project.toml")) &&
           isdir(joinpath(dir, "src", "applications"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && break
        dir = parent
    end
    error("Không tìm thấy thư mục gốc chứa Project.toml")
end

const ROOT_DIR = find_project_root()
Pkg.activate(ROOT_DIR)
include(joinpath(ROOT_DIR, "src", "applications", "applications.jl"))
using .Applications: run_market_basket, run_visual_words
println("Julia version: ", VERSION)
println("Project root:  ", ROOT_DIR)

## Phần 2: Chạy Kiểm Thử Tự Động Chương 3 (`tests/test_ch3_julia.jl`)

In [ ]:
include(joinpath(ROOT_DIR, "tests", "test_ch3_julia.jl"))

## Phần 3: Ứng Dụng 1 — Phân Tích Giỏ Hàng (`Groceries`, `negfin_optimized`)

- Dữ liệu: 9,835 giao dịch, 169 nhóm sản phẩm (`data/chapter3/groceries.csv`).
- Ngưỡng mặc định: `minsup = 0.01`, `minconf = 0.15` (có thể đổi `:optimized` thành `:baseline` để đối chứng).
- Lưu ý: Chỉ số `Lift > 1` phản ánh **tương quan thống kê dương**, không chứng minh quan hệ nhân quả.

In [ ]:
market_res = run_market_basket(;
    minsup=0.01,
    minconf=0.15,
    algo=:optimized,
)

market_dir = joinpath(ROOT_DIR, "results", "chapter3", "market_basket")
println("\n=== MARKET BASKET SUMMARY ===")
println(read(joinpath(market_dir, "summary.txt"), String))
println("=== TOP-10 LUẬT THEO LIFT ===")
for line in Iterators.take(eachline(joinpath(market_dir, "top10_lift.csv")), 11)
    println(line)
end

## Phần 4: Ứng Dụng 2 — Bag of Visual Words (`UCI Digits 8×8`, `negfin_optimized`)

- Trích 9 dense gradient descriptors 6 chiều/ảnh (**không phải SIFT/ORB**), học codebook K-means (`K = 16`, `seed = 42`) trên tập train, khai thác mẫu bằng `negfin_optimized` (`minsup = 0.12`).
- Đánh giá độ làm giàu đặc trưng (*Enrichment*) và độ chính xác *Nearest-Centroid* trên tập test độc lập.

In [ ]:
visual_res = run_visual_words(;
    minsup=0.12,
    words=16,
    seed=42,
    algo=:optimized,
)

visual_dir = joinpath(ROOT_DIR, "results", "chapter3", "visual_words")
println("\n=== VISUAL WORDS SUMMARY ===")
println(read(joinpath(visual_dir, "summary.txt"), String))
println("=== SO SÁNH ĐỘ CHÍNH XÁC NEAREST-CENTROID ===")
println(read(joinpath(visual_dir, "test_accuracy_comparison.csv"), String))